# The divergence of CTTs when evaluated under faults

Models trained without faults can actually get worse as a function of $D$ when evaluated with them. Produces main text Fig. 4 (a)

In [6]:
import os, sys

# Repo root on the path (for `nano_llama` and `scaling_law`), from either cwd.
_HERE = os.getcwd()
for _cand in (_HERE, os.path.dirname(_HERE)):
    if os.path.isdir(os.path.join(_cand, "nano_llama")) and _cand not in sys.path:
        sys.path.insert(0, _cand)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
plt.style.use('default')

from scaling_law import data_loading as dl

# Processed EvalResult dir holding ONE architecture (see the header). Swap for the dedicated sweep's
# eval output when it exists -- nothing below depends on which array the records came from.
EVAL_DIR = "/mnt/storage/eval_summaries/eval_pgrid_512_2026-08-25-19-12-05"   # <-- SET
P_TRAIN = 0.0        # which ROW of the eval grid: the training fault whose checkpoints we follow
K = None             # fault block size; None = require the data to hold exactly one and use it

# sort=False keeps every cohort in record order, so cohorts[p][i] is the SAME model at every
# p_eval. This figure compares durations within an eval condition and does not strictly need
# that, but a per-model overlay or a paired contrast would, and sorting destroys it silently.
cohorts = dl.load_fixed_ptrain_results(EVAL_DIR, p_train=P_TRAIN, k=K, sort=False)
k_used = float(cohorts[next(iter(cohorts))][0].k_eval)

arrs = {pe: dl.eval_cohort_arrays(pts) for pe, pts in cohorts.items()}

# One architecture only: the whole figure rests on N being constant, so it is checked, not assumed.
shapes = {(int(r.model_config.n_embd), int(r.model_config.n_layer))
          for pts in cohorts.values() for r in pts}
if len(shapes) != 1:
    raise ValueError(f"EVAL_DIR holds several architectures {sorted(shapes)}; this figure is the "
                     "fixed-N one, so point it at a dir with a single model shape")
(d_model, n_layer), = shapes
N_ne = float(np.median(arrs[next(iter(arrs))]["N_ne"]))

# Cohorts are RAGGED by construction: every job is scored at the sweep-wide p_eval grid plus
# its own p_train baselines, so a p_eval that is another cohort's baseline exists for only
# part of the model set. A curve over a partial cohort would compare durations that are not
# all present, so the full-coverage grid is computed here and the plot draws from it.
n_max = max(len(v["L"]) for v in arrs.values())
PE_FULL = sorted(pe for pe, v in arrs.items() if len(v["L"]) == n_max)
_partial = sorted(set(arrs) - set(PE_FULL))

_D = arrs[PE_FULL[0]]["D"]
levels = np.array(sorted(set(_D)))
# dl.eval_cohort_arrays names the TOTAL param count "N" (the fit scripts remap it; this
# notebook does not fit, so it keeps the loader's names).
N_tot = float(np.median(arrs[next(iter(arrs))]["N"]))
print(f"d{d_model}_L{n_layer}   N = {N_tot:,.0f} total / {N_ne:,.0f} non-embedding   "
      f"p_train = {P_TRAIN:g}   k = {k_used:g}")
print(f"{n_max} runs per full cohort | {len(PE_FULL)} full p_eval level(s) of {len(arrs)} present")
if _partial:
    print(f"  partial (some models only, not drawn): {', '.join(f'{p:g}' for p in _partial)}")
print()
hdr = f"{'D [tokens]':>13}{'tok/param':>11}{'runs':>6}{'L (clean)':>11}{'L (heaviest)':>14}"
print(hdr); print("-" * len(hdr))
_clean, _heavy = PE_FULL[0], PE_FULL[-1]
for d in levels:
    m0 = arrs[_clean]["D"] == d
    print(f"{d:>13.3g}{d / N_tot:>11.0f}{int(m0.sum()):>6}"
          f"{np.exp(np.mean(np.log(arrs[_clean]['L'][m0]))):>11.3f}"
          f"{np.exp(np.mean(np.log(arrs[_heavy]['L'][arrs[_heavy]['D'] == d]))):>14.3f}")
print(f"\np_eval grid (full coverage): {', '.join(f'{p:g}' for p in PE_FULL)}")
print(f"per-weight rate p_eval/k: {', '.join(f'{p / k_used:.2g}' for p in PE_FULL)}")


d512_L13   N = 49,509,888 total / 41,121,280 non-embedding   p_train = 0   k = 4
63 runs per full cohort | 81 full p_eval level(s) of 81 present

   D [tokens]  tok/param  runs  L (clean)  L (heaviest)
-------------------------------------------------------
     2.48e+08          5     8      3.641         5.499
     4.95e+08         10     8      3.369         5.594
      9.9e+08         20     8      3.186         5.783
     1.98e+09         40     8      3.050         6.022
     3.96e+09         80     8      2.957         6.469
     7.92e+09        160     8      2.883         6.799
     1.58e+10        320     8      2.830         7.606
     3.17e+10        640     7      2.788         8.125

p_eval grid (full coverage): 0, 0.002, 0.00212005, 0.00224731, 0.00238221, 0.0025252, 0.00267678, 0.00283745, 0.00300777, 0.00318832, 0.0033797, 0.00358257, 0.00379762, 0.00402557, 0.00426721, 0.00452335, 0.00479487, 0.00508269, 0.00538778, 0.00571118, 0.006054, 0.0064174, 0.00680261, 0.00721

In [7]:
%matplotlib tk

# ---- the reversal: loss vs training duration, one curve per eval fault --------------------
# Fixed architecture, so this is the raw measurement: each curve is the SAME checkpoints scored
# at a different eval fault, and a curve's tilt IS the token payoff s = dlogL/dlogD there.
# Clean tilts down, heavy tilts up, and the crossing is the result.
#
# Each curve is normalised by its own geometric mean at the shortest duration, so the panel
# shows the TILT rather than the level -- on an absolute axis the faulted curves set the y
# range and squash the clean one flat. Dividing by a constant is an offset in log space, so
# every slope in the table below is identical either way.
SAVE_PATH = None       # e.g. "/home/trevor/figures/reversal_d512.pdf"; None -> just draw it

# Which eval faults to draw: ~5 spanning clean -> heaviest, picked from the full-coverage grid
# by snapping targets evenly spaced in log p so the curves are distinct rather than clustered.
# Past ~7 the ramp steps stop being distinguishable.
N_CURVES = 8
_pos = [p for p in PE_FULL if p > 0]
_targets = np.geomspace(_pos[0], _pos[-1], N_CURVES - 1)
PE_SHOW = [0.0] + [min(_pos, key=lambda p: abs(np.log(p) - np.log(t))) for t in _targets]
PE_SHOW = sorted(set(PE_SHOW))

print(PE_SHOW)

# p_eval is ORDERED, so a single-hue ordinal ramp (light = clean -> dark = heaviest), not categorical
# hues. Light end stops at the 250 step: anything paler fails the contrast floor for a line on white.
from matplotlib.colors import LinearSegmentedColormap
_ramp = LinearSegmentedColormap.from_list("seq_blue_ordinal",
                                          ["#86b6ef", "#3987e5", "#256abf", "#184f95", "#0d366b"])
RAMP = [_ramp(x) for x in np.linspace(0.0, 1.0, len(PE_SHOW))]
INK_SOFT = "#52514e"

# WHAT THE SHADED BAND IS: +/-1 STANDARD ERROR ON THE PLOTTED RATIO -- i.e. the uncertainty of
# the curve itself, NOT the spread of individual runs.
#
# The plotted quantity is R(D) = geomean(L at D) / geomean(L at the shortest D), so its error has
# TWO contributions and both are included:
#
#     Var(log R) = s(D)^2 / n(D)  +  s(D0)^2 / n(D0)
#
# The runs at different D are DISTINCT training runs (each duration is its own max_iters, not one
# run checkpointed repeatedly -- see the table in the cell above), so the two terms are independent
# and simply add: there is no covariance correction, and no bootstrap is needed for a ratio of two
# independent means. Working in logs throughout makes the band multiplicative, which is how these
# losses are compared everywhere else.
#
# This REPLACES a +/-1 sample SD band. That earlier band answered a different question -- "where
# would one more seed land" -- and was up to 2x too wide at short D while being about right at long
# D, because dividing by sqrt(n) happened to be cancelled by adding the reference variance. If the
# seed spread itself is what you want to show, compute np.std(y, ddof=1) and say so in the caption;
# do not read this band as that.
#
# The band is ZERO at the shortest duration by construction: every curve is normalised by its own
# value there, so that ratio is exactly 1 with no uncertainty. A non-zero band at the anchor point
# would be claiming uncertainty in a number that is 1 by definition.
curves, _band_n = {}, []
for pe in PE_SHOW:
    a = arrs[pe]
    logmean, var = [], []
    for d in levels:
        y = np.log(a["L"][a["D"] == d])
        if y.size < 2:
            raise ValueError(
                f"p_eval={pe:g}, D={d:.3g} has {y.size} run(s): a standard error needs at least 2. "
                f"This is the silent-zero-band trap -- the old code drew a confident zero-width band "
                f"for an unreplicated cell. Point EVAL_DIR at a dir with replicates, or drop this "
                f"duration from `levels`.")
        logmean.append(y.mean())
        var.append(y.var(ddof=1) / y.size)      # squared standard error of THIS level's log-mean
        _band_n.append(y.size)
    logmean, var = np.array(logmean), np.array(var)
    m = logmean - logmean[0]                    # log ratio to the curve's own shortest duration
    se = np.sqrt(var + var[0])                  # independent levels -> variances add
    se[0] = 0.0                                 # the anchor ratio is exactly 1, no uncertainty
    curves[pe] = (np.exp(m), np.exp(m - se), np.exp(m + se))

print(f"band = +/-1 standard error of the plotted ratio "
      f"(n = {min(_band_n)}-{max(_band_n)} independent training runs per point)")

# 3.4 in wide, like the other paper figures: the font sizes are the same everywhere, so the
# same width is what makes the text render at the same size once LaTeX scales the column.
fig, ax = plt.subplots(figsize=(3.4, 2.8))
for pe, col in zip(PE_SHOW, RAMP):
    mid, lo, hi = curves[pe]
    ax.fill_between(levels, lo, hi, color=col, alpha=0.18, lw=0, zorder=1)   # +/-1 SE of the ratio
    ax.plot(levels, mid, linestyle="--", marker="o", color=col, lw=1.6, solid_capstyle="round", zorder=3,
            label=f"{pe:.2g}")

ax.set_xscale("log"); ax.set_yscale("log")
# Durations are a handful of discrete values and the loss spans well under a decade, so default log
# ticking is unreadable at this size: tick the actual durations, and put plain numbers on the loss axis.
ax.set_xticks(levels)
ax.set_xticklabels([f"{d / 1e9:.2g}" for d in levels])
ax.xaxis.set_minor_locator(mticker.NullLocator())
# The ratios span well under a decade, so the log axis has no decade boundary to put a major tick on
# and would come out unlabelled. Tick the round ratios that fall inside the data's range instead.
_lo = min(c[1].min() for c in curves.values())
_hi = max(c[2].max() for c in curves.values())
_yt = [t for t in (0.8, 0.9, 0.95, 1, 1.05, 1.1, 1.2, 1.4, 1.6, 1.8, 2, 2.5, 3)
       if _lo * 0.97 <= t <= _hi * 1.03]
ax.set_yticks(_yt)
ax.set_yticklabels([f"{t:g}" for t in _yt])
ax.yaxis.set_minor_locator(mticker.NullLocator())
ax.set_xlabel(r"Training tokens ($D/10^9$)", fontsize=8)
ax.set_ylabel(r"Relative eval loss", fontsize=8)
ax.axhline(1.0, color="0.55", lw=0.7, zorder=2)      # no change from the shortest-duration model
ax.tick_params(labelsize=7.5, width=0.7, length=2.5, colors=INK_SOFT)
ax.grid(True, which="major", axis="both", color="0.88", lw=0.5, zorder=0)
ax.set_axisbelow(True)


for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_linewidth(0.7)

# leg = ax.legend(fontsize=7, title=r"$p_\mathrm{eval}/k$", title_fontsize=7, loc="upper left",
#                 handlelength=1.2, labelspacing=0.25, borderpad=0.4, handletextpad=0.5,
#                 frameon=True, framealpha=0.92, edgecolor="0.75", fancybox=False)
# leg.get_frame().set_linewidth(0.5)

fig.tight_layout(pad=0.3)
if SAVE_PATH:
    # No bbox_inches="tight": tight_layout has set the margins, and "tight" would hand back a figure
    # wider than the column, which LaTeX then rescales.
    fig.savefig(SAVE_PATH, dpi=600)
    print(f"wrote {SAVE_PATH}")

# The effect as a number, from the same measurements: the OLS slope of log L on log D per curve.
# Computed on the RAW losses -- normalising shifts each curve in log space and cannot move its slope.
print(f"d{d_model}_L{n_layer}, p_train = {P_TRAIN:g}: measured token payoff  s = dlogL/dlogD")
hdr = f"{'p_eval':>9}{'p_eval/k':>11}{'s':>9}"
print(hdr); print("-" * len(hdr))
for pe in PE_SHOW:
    a = arrs[pe]
    s = np.polyfit(np.log(a["D"]), np.log(a["L"]), 1)[0]
    print(f"{pe:>9g}{pe / k_used:>11.2g}{s:>+9.3f}")


[0.0, 0.002, 0.004267209053002822, 0.00910453655101462, 0.020591511346250254, 0.043934141815864705, 0.09373808384628385, 0.2]
band = +/-1 standard error of the plotted ratio (n = 7-8 independent training runs per point)
d512_L13, p_train = 0: measured token payoff  s = dlogL/dlogD
   p_eval   p_eval/k        s
-----------------------------
        0          0   -0.053
    0.002     0.0005   -0.045
0.00426721     0.0011   -0.034
0.00910454     0.0023   -0.013
0.0205915     0.0051   +0.026
0.0439341      0.011   +0.072
0.0937381      0.023   +0.096
      0.2       0.05   +0.082
